# PWV02 : Explore statistical uncertainties from the fits

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS
- creation date : 2025-10-29 (v1), adapted to the 2026 merged files on 2026-11-13
- kernel : conda_py313 (python 3.13)
- input : merged Spectractor files of `../2026-11-13-SpectractorExtraction-FromButler` (selected in `PWV00_parameters.py`) and the standard star magnitudes `../2025-10-29-TOOLS/data/targets_magnitudes.csv`

**Goal** : study the statistical error on PWV returned by the Spectractor fits, as a function of the target, its magnitude and colors, the chi2 of the fit and the airmass, for the two fit methods :
- `ram` : fit of the spectrogram (2D image of the CCD),
- `rum` : fit of the 1D spectrum (projection on the dispersion axis).

Then estimate, night by night, the repeatability of PWV from the residual to a local linear interpolation between the neighbouring points (OG550 data).

The errors are used **as returned by the fits** : the PWV repeatability term is NOT added here.

The cuts and the quality exploration (filters, targets, chi2, ...) are done in `PWV01_ExploreHoloQualityPWV.ipynb`.

In [ ]:
import os
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.gridspec import GridSpec
import seaborn as sns

warnings.simplefilter('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

plt.rcParams.update({
    'figure.figsize': (16, 8),
    'axes.labelsize': 'xx-large',
    'axes.titlesize': 'xx-large',
    'xtick.labelsize': 'xx-large',
    'ytick.labelsize': 'xx-large',
    'legend.fontsize': 'xx-large',
})
props = dict(boxstyle='round', facecolor='white', alpha=0.1)

print(f'python {python_version()}, numpy {np.__version__}, pandas {pd.__version__}')

## Configuration

In [ ]:
from PWV00_parameters import (
    version_run, atmfilename, tag,
    FLAG_PWVFILTERS, PWV_FILTER_LIST, PWV_FILTEROG550_LIST,
    PWV_METHOD, PWV_COL, PWV_ERR_COL,
    CHI2_COL, D2CCD_COL, TARGET_COL, FILTER_COL, TIME_COL,
    SIGMA_PWV_REPEAT,
    site_lsst, get_astronomical_midnight, getSelectionCut, DumpConfig,
)
DumpConfig()

In [ ]:
NB_TAG = 'PWV02_StatErrors'
prefix = 'pwv02_staterr'
pathfigs = f'figs_{NB_TAG}'
pathdata = f'data_{NB_TAG}'
for path in (pathfigs, pathdata):
    os.makedirs(path, exist_ok=True)


def save_fig(fig, name, formats=('png', 'pdf')):
    '''Save a figure in the figure directory, with the notebook prefix.'''
    for ext in formats:
        fig.savefig(f'{pathfigs}/{prefix}_{name}.{ext}', bbox_inches='tight')

In [ ]:
PWVMIN, PWVMAX = 0., 18.          # y-range of the PWV plots [mm]

# the collimator was installed on 2023-09-30 : keep only the nights after
FLAG_WITHCOLLIMATOR = True
DATE_WITHCOLLIMATOR = 20230930

PLOT_EACH_NIGHT = True            # make one figure per night (saved in PNG)
N_NIGHTS_DISPLAYED = 3            # number of per-night figures displayed in the notebook (all are saved)
MIN_OBS_FIT = 9                   # minimum number of points in a night for the fit and the interpolation

TARGETS_MAG_FILE = '../2025-10-29-TOOLS/data/targets_magnitudes.csv'

# the two fit methods : suffix of the columns in the merged files
METHODS = {'ram': 'spectrogram', 'rum': 'spectrum'}
ERR_YLIM = {'ram': 0.08, 'rum': 0.20}      # y-range of the plots of the PWV error [mm]

FILTER_COLORS = {'empty': 'blue',
                 'BG40_65mm_1': 'green',
                 'BG40': 'green',
                 'OG550_65mm_1': 'red',
                 'OG550': 'red',
                 'FELH0600': 'purple',
                 }
# the two filters used in the study of the errors
FILTERS_STUDIED = [('OG550_65mm_1', 'OG550'), ('empty', 'empty')]

## Read the merged file and apply the selection

In [ ]:
df_spec = pd.DataFrame(np.load(atmfilename, allow_pickle=True))
df_spec['Time'] = pd.to_datetime(df_spec[TIME_COL], utc=True)
df_spec['nightObs'] = df_spec['id'] // 100_000
df_spec = df_spec.sort_values(by='Time')

unknown = set(df_spec[FILTER_COL].unique()) - set(FILTER_COLORS)
assert not unknown, f'add these filters to FILTER_COLORS : {unknown}'

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec[FILTER_COL].isin(PWV_FILTER_LIST)]
if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['nightObs'] > DATE_WITHCOLLIMATOR]

# quality cuts
df_sel = df_spec[getSelectionCut(df_spec)].reset_index(drop=True)
print(f'{len(df_sel)} / {len(df_spec)} spectra kept after the quality cuts')
df_sel.groupby(FILTER_COL).size()

## Statistical error on PWV
### Standard star magnitudes
The magnitudes and colors of the standard stars in the LSST filters have been computed in `../2025-10-29-TOOLS`.

In [ ]:
df_mag = pd.read_csv(TARGETS_MAG_FILE, index_col=0)
df_mag.index = df_mag.index.str.replace(' ', '')       # 'HD 185975' -> 'HD185975'
df_mag = df_mag[~df_mag.index.duplicated()].sort_values(by='y')

targets_in_data = set(df_sel[TARGET_COL].unique())
missing = sorted(targets_in_data - set(df_mag.index))
if missing:
    print('WARNING : targets without magnitudes (not shown in the plots per target) :', missing)

# targets ordered by increasing y magnitude, and one color per target
targets = [t for t in df_mag.index if t in targets_in_data]
color_map = dict(zip(targets, sns.color_palette('tab20', n_colors=len(targets))))
print(targets)

### Distribution of the errors

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for ax, (method, label) in zip(axs, METHODS.items()):
    errcol = f'PWV [mm]_err_{method}'
    rng = (0., ERR_YLIM[method])
    ax.hist(df_sel[errcol], bins=50, range=rng, color='k', histtype='step', label='all filters')
    for filt, group in df_sel.groupby(FILTER_COL):
        ax.hist(group[errcol], bins=50, range=rng, color=FILTER_COLORS[filt], alpha=0.5, label=filt)
    ax.legend()
    ax.set_title(f'PWV statistical error in {label} ({method})')
    ax.set_xlabel('PWV err (mm)')
plt.suptitle(f'Statistical error on PWV, {tag}')
plt.tight_layout()
save_fig(fig, 'pwverr_hist')
plt.show()

# median error per filter and method
summary_err = pd.DataFrame({m: df_sel.groupby(FILTER_COL)[f'PWV [mm]_err_{m}'].median() for m in METHODS})
summary_err.columns = [f'median err {m} [mm]' for m in METHODS]
summary_err

### Dispersion of the errors per target

In [ ]:
def boxplot_per_target(ax, df, ycol, ylim=None, title=''):
    '''Box plot and points of ycol per target, targets ordered by magnitude.'''
    sns.boxplot(data=df, x=TARGET_COL, y=ycol, hue=TARGET_COL, order=targets, hue_order=targets,
                palette=color_map, legend=False, fliersize=0, ax=ax)
    sns.stripplot(data=df, x=TARGET_COL, y=ycol, order=targets, color='black', alpha=0.4, size=2, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=90)
    if ylim is not None:
        ax.set_ylim(*ylim)


fig, ax = plt.subplots(figsize=(20, 6))
boxplot_per_target(ax, df_sel, CHI2_COL, title='CHI2_FIT per target')
ax.set_yscale('log')
save_fig(fig, 'chi2fit_vs_target')
plt.show()

In [ ]:
fig, axs = plt.subplots(len(METHODS), 1, figsize=(20, 5 * len(METHODS)), layout='constrained')
for ax, (method, label) in zip(axs, METHODS.items()):
    boxplot_per_target(ax, df_sel, f'PWV [mm]_err_{method}', (0., ERR_YLIM[method]),
                       f'PWV errors per target in {label} ({method})')
    ax.set_ylabel('PWV error [mm]')
save_fig(fig, 'pwverr_vs_target_ramrum')
plt.show()

In [ ]:
# one row per filter, one column per method
fig, axs = plt.subplots(len(FILTERS_STUDIED), len(METHODS), figsize=(24, 5 * len(FILTERS_STUDIED)), layout='constrained')
for i, (filt, filtname) in enumerate(FILTERS_STUDIED):
    df_f = df_sel[df_sel[FILTER_COL] == filt]
    for j, (method, label) in enumerate(METHODS.items()):
        boxplot_per_target(axs[i, j], df_f, f'PWV [mm]_err_{method}', (0., ERR_YLIM[method]),
                           f'PWV errors per target, filter {filtname}, {label} ({method})')
        axs[i, j].set_ylabel('PWV error [mm]')
save_fig(fig, 'pwverr_vs_target_perfilter')
plt.show()

### Relation of the error with magnitude, colors, chi2 and airmass
For each target : individual measurements (small points) and median (large point).

In [ ]:
rng_jitter = np.random.default_rng(0)


def plot_by_target(ax, df, ycol, xspec, ylim, jitter=0., annotate=False):
    '''Scatter of ycol for each target.
    xspec is either a column of df_mag (magnitude or color : one x per target, with a horizontal jitter),
    or a column of df (one x per measurement).'''
    per_target_x = xspec in df_mag.columns
    for target in targets:
        sub = df[df[TARGET_COL] == target]
        if len(sub) == 0:
            continue
        y = sub[ycol].values
        if per_target_x:
            xmed = df_mag.loc[target, xspec]
            x = xmed + rng_jitter.normal(0., jitter, len(y))
        else:
            x = sub[xspec].values
            xmed = np.median(x)
        ymed = np.median(y)
        ax.scatter(x, y, color=color_map[target], alpha=0.5, s=60)
        ax.scatter(xmed, ymed, color=color_map[target], s=150, edgecolor='black', zorder=5, label=target)
        if annotate and ymed < 0.9 * ylim:
            ax.text(xmed, ymed + 0.03 * ylim, f'{ymed:.3f}', color=color_map[target], ha='center', va='bottom', fontsize=12)
    ax.set_ylim(0., ylim)
    ax.grid(True, alpha=0.3, linestyle='--')


def grid_by_target(xspec, xlabel, name, jitter=0., annotate=False):
    '''PWV error versus xspec : one row per filter, one column per method.'''
    fig, axs = plt.subplots(len(FILTERS_STUDIED), len(METHODS), figsize=(18, 6 * len(FILTERS_STUDIED)), layout='constrained')
    for i, (filt, filtname) in enumerate(FILTERS_STUDIED):
        df_f = df_sel[df_sel[FILTER_COL] == filt]
        for j, (method, label) in enumerate(METHODS.items()):
            ax = axs[i, j]
            plot_by_target(ax, df_f, f'PWV [mm]_err_{method}', xspec, ERR_YLIM[method], jitter, annotate)
            ax.set_xlabel(xlabel)
            ax.set_ylabel('PWV error [mm]')
            ax.set_title(f'filter {filtname}, {label} ({method})', fontsize=14)
    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside right upper', title='Targets', fontsize=12)
    save_fig(fig, name)
    plt.show()


grid_by_target('y', 'Magnitude y (mag)', 'pwverr_vs_mag', jitter=0.02, annotate=True)

In [ ]:
grid_by_target(CHI2_COL, 'CHI2_FIT', 'pwverr_vs_chi2fit')

In [ ]:
grid_by_target('AIRMASS', 'airmass', 'pwverr_vs_airmass')

In [ ]:
SedColors = ['u-g', 'g-r', 'r-i', 'i-z', 'z-y']

for method, label in METHODS.items():
    fig, axs = plt.subplots(len(SedColors), len(FILTERS_STUDIED), figsize=(18, 4 * len(SedColors)), layout='constrained')
    for i, sedcolor in enumerate(SedColors):
        for j, (filt, filtname) in enumerate(FILTERS_STUDIED):
            ax = axs[i, j]
            df_f = df_sel[df_sel[FILTER_COL] == filt]
            plot_by_target(ax, df_f, f'PWV [mm]_err_{method}', sedcolor, ERR_YLIM[method], jitter=0.003, annotate=True)
            ax.set_xlabel(f'{sedcolor} (mag)')
            ax.set_ylabel('PWV error [mm]')
            ax.set_title(f'PWV error vs {sedcolor}, filter {filtname}, {label} ({method})', fontsize=14)
    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside right upper', title='Targets', fontsize=10)
    save_fig(fig, f'pwverr_vs_colors_{method}')
    plt.show()

In [ ]:
# chi2 of the fit versus airmass, per target
fig, axs = plt.subplots(1, len(FILTERS_STUDIED), figsize=(18, 6), layout='constrained')
for ax, (filt, filtname) in zip(axs, FILTERS_STUDIED):
    plot_by_target(ax, df_sel[df_sel[FILTER_COL] == filt], CHI2_COL, 'AIRMASS', df_sel[CHI2_COL].max() * 1.05)
    ax.set_xlabel('airmass')
    ax.set_ylabel(CHI2_COL)
    ax.set_title(f'CHI2_FIT vs airmass, filter {filtname}', fontsize=14)
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside right upper', title='Targets', fontsize=10)
save_fig(fig, 'chi2fit_vs_airmass')
plt.show()

## Night by night : repeatability from the local linear interpolation (OG550)
For each point `i` of a night, the PWV is interpolated linearly between the points `i-1` and `i+1`. The residual `dz = PWV[i] - z[i]` has a variance `sigma^2 * (1 + (1-alpha)^2 + alpha^2)` where `alpha` is the relative time position of the point `i` between its neighbours and `sigma` the PWV repeatability.
We estimate `sigma` from the robust dispersion of `dz` (`sig_iqr`), and the expected dispersion of the difference between two consecutive points is `sqrt(2) * sigma`.

Note : in the 2025 version the factor between `sig_iqr(dz)` and the dispersion of consecutive differences was `sqrt(2)`, which assumes that `dz` has the dispersion of one measurement ; the factor above takes the noise of the two neighbours into account.

The PWV method (`ram` or `rum`) is the one selected by `PWV_METHOD` in `PWV00_parameters.py`.

In [ ]:
def std_iqr(x):
    '''Robust standard deviation from the inter-quartile range (IQR / 1.349 for a normal distribution).'''
    q25, q75 = np.quantile(x, [0.25, 0.75])
    return (q75 - q25) / 1.349


def delta_angle_deg(a1, a2):
    '''Angle difference in degrees, brought back to [-180, 180].'''
    return (a1 - a2 + 180.) % 360. - 180.


def fit_line(x, y, yerr):
    '''Weighted least squares fit of y = a + b x. Returns (a, b), their errors, and chi2/ndof.
    The covariance matrix is rescaled by chi2/ndof.'''
    w = 1. / yerr**2
    A = np.vstack([np.ones_like(x), x]).T
    cov = np.linalg.inv(A.T @ (A * w[:, None]))
    params = cov @ (A.T @ (w * y))
    chi2dof = np.sum(((y - A @ params) / yerr)**2) / (len(x) - 2)
    cov *= chi2dof
    return params, np.sqrt(np.diag(cov)), chi2dof


def local_linear_interp(t, y, ey):
    '''For each point i (except the first and the last), linear interpolation z[i] between the points i-1 and i+1.
    t : datetime64 array. Returns t, z, dz = y - z, the error on z and alpha = (t[i]-t[i-1]) / (t[i+1]-t[i-1]).'''
    tsec = (t - t[0]) / np.timedelta64(1, 's')
    with np.errstate(divide='ignore', invalid='ignore'):
        alpha = (tsec[1:-1] - tsec[:-2]) / (tsec[2:] - tsec[:-2])
    z = y[:-2] + (y[2:] - y[:-2]) * alpha
    ez = np.sqrt((1 - alpha)**2 * ey[:-2]**2 + alpha**2 * ey[2:]**2)
    dz = y[1:-1] - z
    ok = np.isfinite(dz)
    return t[1:-1][ok], z[ok], dz[ok], ez[ok], alpha[ok]


def get_midnight(night):
    '''Astronomical midnight (UTC timestamp) of the night YYYYMMDD.'''
    day = datetime.strptime(str(night), '%Y%m%d').date()
    ts = pd.Timestamp(get_astronomical_midnight(site_lsst, day))
    return ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')


# periods of stability (hours with respect to midnight), selected by eye on the 2025 nights
dict_select_stablemoments = {
    20250127: {'tstart': 2.1, 'tstop': 4.},
    20250224: {'tstart': -1.9, 'tstop': 0.1},
    20250226: {'tstart': -1., 'tstop': 0.},
    20250303: {'tstart': -1., 'tstop': -0.1},
    20250311: {'tstart': -2.1, 'tstop': -1.1},
    20250317: {'tstart': 1., 'tstop': 5.2},
    20250319: {'tstart': 0.5, 'tstop': 5.2},
    20250320: {'tstart': 0.6, 'tstop': 5.2},
    20250402: {'tstart': -4., 'tstop': -2.2},
    20250407: {'tstart': -1., 'tstop': 0.},
    20250913: {'tstart': 1., 'tstop': 3.},
    20250917: {'tstart': 0.5, 'tstop': 3.},
}

In [ ]:
AZ_COL = 'AZ'     # azimuth of the telescope [deg], in [-180, 180]


def add_airmass_azimuth(ax, x, airmass, az):
    '''Two additional y axes (airmass, azimuth) sharing the x axis of ax.'''
    ax_air = ax.twinx()
    ax_az = ax.twinx()
    ax_az.spines.right.set_position(('axes', 1.1))
    p_air = ax_air.scatter(x, airmass, color='purple', marker='x', label='airmass')
    p_az = ax_az.scatter(x, az, color='darkblue', marker='+', label='azimuth')
    ax_air.set_ylim(2.5, 0.5)
    ax_air.set_ylabel('airmass', color='purple')
    ax_az.set_ylabel('azimuth (deg)', color='darkblue')
    ax_air.tick_params(axis='y', colors='purple')
    ax_az.tick_params(axis='y', colors='darkblue')
    ax_az.legend(handles=[p_air, p_az], loc='lower right')


def plot_night_stat(d, night, midnight, st, interp, diffs, display):
    '''4 panels for one night : PWV vs time with the interpolation, histogram of dz,
    consecutive PWV differences vs time, |dPWV| vs |dAz|.'''
    tout, zout, dzout, ezout = interp
    dt_hr, dpwv2, edpwv, daz, dtt = diffs
    t = d['Time']
    y = d[PWV_COL].values
    sig_dpwv = st['sig_dpwv']

    fig = plt.figure(figsize=(20, 13), layout='constrained')
    gs = GridSpec(2, 2, width_ratios=[2, 1], figure=fig)
    ax = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    # top left : PWV versus time
    ax.errorbar(t, y, yerr=d[PWV_ERR_COL], fmt='o', color='red', ecolor='k', capsize=5, markersize=10, label='OG550')
    ax.errorbar(tout, zout, yerr=ezout, fmt='+', color='darkgreen', capsize=5, markersize=5, label='interpolated')
    for i in range(1, len(t) - 1):
        ax.plot([t.iloc[i - 1], t.iloc[i + 1]], [y[i - 1], y[i + 1]], ':', color='g', lw=0.8)
    ax.axvline(midnight, color='k', ls=':', lw=2)
    if night in dict_select_stablemoments:
        p = dict_select_stablemoments[night]
        ax.axvspan(midnight + pd.Timedelta(hours=p['tstart']), midnight + pd.Timedelta(hours=p['tstop']), color='red', alpha=0.05)
    ax.set_ylim(PWVMIN, PWVMAX)
    ax.set_ylabel(PWV_COL)
    ax.set_xlabel('time (UTC)')
    ax.xaxis.set_major_formatter(DateFormatter('%y-%m-%dT%H:%M'))
    ax.tick_params(axis='x', labelrotation=45)
    ax.grid()
    ax.legend(loc='upper right')
    add_airmass_azimuth(ax, t, d['AIRMASS'], d[AZ_COL])
    text1 = '\n'.join((f"count : {st['count']}", f"mean : {st['mean']:.1f} mm",
                       f"median : {st['median']:.1f} mm", f"std : {st['std']:.1f} mm"))
    text2 = '\n'.join(('d(PWV)/dt :', f"slope : {st['slope']:.3f} mm/h", f"slope err : {st['slope_err']:.3f} mm/h"))
    ax.text(0.03, 0.95, text1, transform=ax.transAxes, fontsize=14, va='top', bbox=props)
    ax.text(0.35, 0.95, text2, transform=ax.transAxes, fontsize=14, va='top', bbox=props)

    # top right : residuals to the local interpolation
    ax2.hist(dzout, bins=50, range=(-2, 2), facecolor='b')
    ax2.set_xlabel(r'$\Delta PWV$ (mm)')
    text4 = '\n'.join((r'$\Delta PWV$ = PWV - interpolation :', f"median : {st['dz_median']:.3f} mm",
                       f"sig_iqr : {st['dz_sigiqr']:.3f} mm", f"sigma(PWV) : {st['sig_pwv']:.3f} mm"))
    ax2.text(0.01, 0.95, text4, transform=ax2.transAxes, fontsize=14, va='top', bbox=props)

    # bottom left : difference between consecutive points, colored by the azimuth change
    ax3.errorbar(dt_hr, dpwv2, yerr=edpwv, fmt='none', ecolor='gray', alpha=0.5, capsize=3)
    sc3 = ax3.scatter(dt_hr, dpwv2, c=daz, cmap='jet', s=100, zorder=3)
    fig.colorbar(sc3, ax=ax3, location='bottom', pad=0.2, aspect=40, label='|ΔAzimuth| [deg]')
    for k, ls in ((1, ':'), (2, '-.'), (3, '--')):
        ax3.axhline(k * sig_dpwv, color='k', ls=ls)
        ax3.axhline(-k * sig_dpwv, color='k', ls=ls)
    ax3.axvline(0., color='k', ls=':', lw=2)
    ax3.axhline(0., color='r', lw=2)
    if night in dict_select_stablemoments:
        p = dict_select_stablemoments[night]
        ax3.axvspan(p['tstart'], p['tstop'], color='red', alpha=0.05)
    ax3.set_ylim(-2., 2.)
    ax3.set_ylabel(r'$\Delta PWV$ [mm]')
    ax3.set_xlabel('Relative time wrt midnight (hour)')
    ax3.grid()
    add_airmass_azimuth(ax3, dt_hr, d['AIRMASS'].values[1:], d[AZ_COL].values[1:])

    # bottom right : |dPWV| versus |dAz|, colored by the time difference
    ax4.errorbar(daz, np.abs(dpwv2), yerr=edpwv, fmt='none', ecolor='gray', alpha=0.5, capsize=2)
    sc4 = ax4.scatter(daz, np.abs(dpwv2), c=dtt, cmap='jet', s=100, zorder=3)
    fig.colorbar(sc4, ax=ax4, label='Δt (min)')
    for k, ls in ((1, ':'), (2, '-.'), (3, '--')):
        ax4.axhline(k * sig_dpwv, color='k', ls=ls, label=rf'${k} \sigma$')
    ax4.set_ylim(0., np.abs(dpwv2).max() * 1.2)
    ax4.legend(loc='upper right')
    ax4.set_title('azimuth distance')
    ax4.set_xlabel(r'$|\Delta Az|$ (deg)')
    ax4.set_ylabel(r'$|\Delta PWV|$ (mm)')

    plt.suptitle(f'night {night}, {tag}', fontsize=16)
    save_fig(fig, f'per_night_{night}', formats=('png',))
    if display:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
df_og = df_sel[df_sel[FILTER_COL].isin(PWV_FILTEROG550_LIST)]
print(f'{len(df_og)} OG550 spectra in {df_og.nightObs.nunique()} nights')

rows = []
n_displayed = 0
for night, d in df_og.groupby('nightObs'):
    d = d.sort_values('Time')
    n = len(d)
    pwv = d[PWV_COL]
    st = {'nightObs': night, 'count': n, 'mean': pwv.mean(), 'median': pwv.median(), 'std': pwv.std(),
          'slope': np.nan, 'slope_err': np.nan, 'chi2dof': np.nan,
          'dz_median': np.nan, 'dz_sigiqr': np.nan, 'sig_pwv': np.nan, 'sig_dpwv': np.nan}
    if n >= MIN_OBS_FIT:
        y = pwv.values
        ey = d[PWV_ERR_COL].values
        # straight line fit
        x = ((d['Time'] - d['Time'].min()).dt.total_seconds() / 3600.).values
        try:
            params, perr, chi2dof = fit_line(x, y, ey)
            st.update(slope=params[1], slope_err=perr[1], chi2dof=chi2dof)
        except np.linalg.LinAlgError:
            pass
        # local linear interpolation and repeatability estimate
        tout, zout, dzout, ezout, alpha = local_linear_interp(d['Time'].values, y, ey)
        if len(dzout) >= 3:
            sigiqr = std_iqr(dzout)
            sig_pwv = sigiqr / np.sqrt(1. + np.mean((1 - alpha)**2 + alpha**2))
            st.update(dz_median=np.median(dzout), dz_sigiqr=sigiqr, sig_pwv=sig_pwv, sig_dpwv=np.sqrt(2.) * sig_pwv)
            if PLOT_EACH_NIGHT:
                midnight = get_midnight(night)
                # differences between consecutive points
                dt_hr = ((d['Time'] - midnight) / pd.Timedelta(hours=1)).values
                diffs = (dt_hr[1:], np.diff(y), np.sqrt(ey[1:]**2 + ey[:-1]**2),
                         np.abs(delta_angle_deg(d[AZ_COL].values[1:], d[AZ_COL].values[:-1])), np.diff(dt_hr) * 60.)
                plot_night_stat(d, night, midnight, st, (tout, zout, dzout, ezout), diffs, display=(n_displayed < N_NIGHTS_DISPLAYED))
                n_displayed += 1
    rows.append(st)

df_night = pd.DataFrame(rows).set_index('nightObs')
df_night['Time'] = pd.to_datetime(df_night.index.astype(str), format='%Y%m%d')
df_night.to_csv(f'{pathdata}/{prefix}_nightly_summary_{PWV_METHOD}_{version_run}.csv')
print(f'{len(df_night)} nights, {df_night.sig_pwv.notna().sum()} with a repeatability estimate')
df_night.head()

## Summary of the night quality

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
ax1, ax2, ax3 = axs

df_night['std'].hist(ax=ax1, bins=100, range=(0, 2), facecolor='b')
ax1.set_xlim(0., 1.)
ax1.set_xlabel('$RMS(PWV)$ (mm)')
ax1.set_title('per-night dispersion')

ax2.scatter(df_night['median'], df_night['std'], marker='o', color='b')
ax2.set_xlabel('$median(PWV)$ (mm)')
ax2.set_ylabel('$RMS(PWV)$ (mm)')
ax2.set_title('per-night dispersion')
ax2.grid(True)

ax3.errorbar(df_night['median'], df_night['slope'], xerr=df_night['std'], yerr=df_night['slope_err'],
             fmt='o', color='b', ecolor='gray', capsize=3, markersize=8)
ax3.set_ylim(-0.5, 0.5)
ax3.set_xlabel('$median(PWV)$ (mm)')
ax3.set_ylabel('$fitted slope dPWV/dt $ (mm/h)')
ax3.set_title('fitted drift per night')
ax3.grid(True)

plt.suptitle(f'PWV with OG550 data, method {PWV_METHOD}, {tag}')
plt.tight_layout()
save_fig(fig, 'nightly_dispersion_and_slope')
plt.show()

In [ ]:
date_form = DateFormatter('%y-%m-%d')
fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
ax1, ax2 = axs

ax1.errorbar(df_night['Time'], df_night['median'], yerr=df_night['std'], fmt='o', ms=5, color='r', ecolor='k')
ax1.set_ylabel('PWV (mm)')
ax1.set_title(f'Median Precipitable water vapor per night, {tag}')
ax1.set_ylim(PWVMIN, PWVMAX)
ax1.grid()

ax2.errorbar(df_night['Time'], df_night['slope'], yerr=df_night['slope_err'], fmt='.', color='r', ecolor='k', ms=10)
ax2.set_title('Fitted drift per night')
ax2.set_ylabel('dPWV/dt (mm/h)')
ax2.set_xlabel('time')
ax2.set_ylim(-1., 1.)
ax2.grid()
ax2.xaxis.set_major_formatter(date_form)
ax2.tick_params(axis='x', labelrotation=45)

plt.tight_layout()
save_fig(fig, 'pwv_medians_and_slopes_allnights')
plt.show()

In [ ]:
# repeatability of PWV estimated night by night
sig = df_night['sig_pwv'].dropna()
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
ax1, ax2 = axs

ax1.hist(sig, bins=50, range=(0., 0.6), facecolor='b')
ax1.axvline(sig.median(), color='k', ls='-', label=f'median = {sig.median():.3f} mm')
ax1.axvline(SIGMA_PWV_REPEAT, color='r', ls='--', label=f'SIGMA_PWV_REPEAT = {SIGMA_PWV_REPEAT} mm')
ax1.set_xlabel(r'$\sigma(PWV)$ from local interpolation (mm)')
ax1.set_title('per-night PWV repeatability')
ax1.legend(fontsize=14)

ax2.scatter(df_night['median'], df_night['sig_pwv'], color='b')
ax2.axhline(SIGMA_PWV_REPEAT, color='r', ls='--')
ax2.set_xlabel('$median(PWV)$ (mm)')
ax2.set_ylabel(r'$\sigma(PWV)$ (mm)')
ax2.set_ylim(0., 0.6)
ax2.grid(True)

plt.suptitle(f'PWV repeatability, OG550, method {PWV_METHOD}, {tag}')
plt.tight_layout()
save_fig(fig, 'nightly_repeatability')
plt.show()
print(f'median sigma(PWV) = {sig.median():.3f} mm, 16%-84% : {sig.quantile(0.16):.3f} - {sig.quantile(0.84):.3f} mm ({len(sig)} nights)')